# MGMT298D: Science and Strategy of AI
## Week 1: Linear Regression & Regularization
### UCLA Anderson School of Management

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style('whitegrid')

## Load Data

In [ ]:
# Load H&M sales data
url = "https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/HMData.csv"
df = pd.read_csv(url)

print(f"Full dataset: {len(df)} rows, {df.shape[1]} columns")
print(f"\nAvailable products ({df['name'].nunique()}):")
for name in sorted(df['name'].unique()):
    count = len(df[df['name'] == name])
    print(f"  {name} ({count} rows)")

## Select a Product

In [ ]:
# Change this string to pick a different product
PRODUCT = "Vest top"

df_product = df[df['name'] == PRODUCT].reset_index(drop=True)
print(f"Product: {PRODUCT}")
print(f"Rows: {len(df_product)}")
print(f"\nSales range: {df_product['sales'].min()} – {df_product['sales'].max()}")
print(f"Price range: {df_product['price'].min():.4f} – {df_product['price'].max():.4f}")
df_product[['id', 'sales', 'price']].head(10)

## Explore: Sales vs Price

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df_product['price'], df_product['sales'], alpha=0.4, s=30)
plt.xlabel('Price')
plt.ylabel('Sales')
plt.title(f'{PRODUCT}: Sales vs Price')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 1: OLS with Price Only

In [ ]:
# Start simple — can price alone predict sales?
features_v1 = ['price']

X = df_product[features_v1].values
y = df_product['sales'].values

# Time-based 80/20 split
split = int(0.8 * len(df_product))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

scaler_v1 = StandardScaler()
X_train_s = scaler_v1.fit_transform(X_train)
X_test_s = scaler_v1.transform(X_test)

ols_v1 = LinearRegression()
ols_v1.fit(X_train_s, y_train)
pred_v1 = ols_v1.predict(X_test_s)

mae_v1 = mean_absolute_error(y_test, pred_v1)
r2_v1 = r2_score(y_test, pred_v1)

print(f"OLS (price only)")
print(f"  MAE:  {mae_v1:.2f}")
print(f"  R²:   {r2_v1:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, pred_v1, alpha=0.5, s=40)
mn, mx = min(y_test.min(), pred_v1.min()), max(y_test.max(), pred_v1.max())
plt.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect')
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title(f'OLS (Price Only) — MAE: {mae_v1:.0f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 2: Feature Engineering
Generate a large number of features from price and sales history: lags, rolling statistics, polynomial terms, and interaction terms.

In [ ]:
# --- Lag features ---
for i in range(1, 7):
    df_product[f'lag_{i}'] = df_product['sales'].shift(i)

# --- Rolling statistics ---
for w in [3, 5, 7]:
    df_product[f'ma_{w}'] = df_product['sales'].rolling(w).mean().shift(1)
    df_product[f'std_{w}'] = df_product['sales'].rolling(w).std().shift(1)

# --- Price features ---
df_product['price_change'] = df_product['price'].diff()
df_product['price_pct_change'] = df_product['price'].pct_change()

# --- Squared terms ---
df_product['price_sq'] = df_product['price'] ** 2
for i in range(1, 4):
    df_product[f'lag_{i}_sq'] = df_product[f'lag_{i}'] ** 2

# --- Interaction terms: price × lags ---
for i in range(1, 5):
    df_product[f'price_x_lag_{i}'] = df_product['price'] * df_product[f'lag_{i}']

# --- Interaction terms: price × rolling ---
for w in [3, 5, 7]:
    df_product[f'price_x_ma_{w}'] = df_product['price'] * df_product[f'ma_{w}']

# --- Cross-lag interactions ---
df_product['lag1_x_lag2'] = df_product['lag_1'] * df_product['lag_2']
df_product['lag1_x_lag3'] = df_product['lag_1'] * df_product['lag_3']
df_product['lag2_x_lag3'] = df_product['lag_2'] * df_product['lag_3']

# --- Momentum ratios (safe division) ---
df_product['lag1_over_lag2'] = df_product['lag_1'] / df_product['lag_2'].replace(0, np.nan)
df_product['lag1_over_ma3'] = df_product['lag_1'] / df_product['ma_3'].replace(0, np.nan)

# --- Fill NaN from shifts, rolling windows, and division ---
df_product.fillna(0, inplace=True)
df_product.replace([np.inf, -np.inf], 0, inplace=True)

# Collect all engineered feature names (everything except id, sales, name, and one-hot cols)
exclude = {'id', 'sales', 'name', 'price'}
features_v2 = [c for c in df_product.columns if c not in exclude
               and df_product[c].dtype in ['float64', 'int64']
               and df_product[c].std() > 0]

# Also keep price as first feature
features_v2 = ['price'] + [f for f in features_v2 if f != 'price']

print(f"Total engineered features: {len(features_v2)}")
print(f"\nFeature list:")
for i, f in enumerate(features_v2, 1):
    print(f"  {i:2d}. {f}")

## Step 3: OLS with All Engineered Features

In [ ]:
# Fit OLS on the full engineered feature set
X2 = df_product[features_v2].values
y2 = df_product['sales'].values

X2_train, X2_test = X2[:split], X2[split:]
y2_train, y2_test = y2[:split], y2[split:]

scaler_v2 = StandardScaler()
X2_train_s = scaler_v2.fit_transform(X2_train)
X2_test_s = scaler_v2.transform(X2_test)

ols_v2 = LinearRegression()
ols_v2.fit(X2_train_s, y2_train)
pred_v2 = ols_v2.predict(X2_test_s)

mae_v2 = mean_absolute_error(y2_test, pred_v2)
r2_v2 = r2_score(y2_test, pred_v2)

print(f"OLS ({len(features_v2)} features)")
print(f"  MAE:  {mae_v2:.2f}")
print(f"  R²:   {r2_v2:.4f}")
print(f"\nImprovement over price-only: MAE dropped by {mae_v1 - mae_v2:.2f}")

In [ ]:
# Side-by-side: price-only vs full feature set
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, pred_v1, alpha=0.5, s=40)
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=2)
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Price Only (1 feature) — MAE: {mae_v1:.0f}')
axes[0].grid(True, alpha=0.3)

mn2, mx2 = min(y2_test.min(), pred_v2.min()), max(y2_test.max(), pred_v2.max())
axes[1].scatter(y2_test, pred_v2, alpha=0.5, s=40, color='green')
axes[1].plot([mn2, mx2], [mn2, mx2], 'r--', lw=2)
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title(f'All {len(features_v2)} Features — MAE: {mae_v2:.0f}')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## OLS Coefficients (Top 15 by Magnitude)

In [ ]:
# With many features, show the largest coefficients
coef_all = pd.DataFrame({
    'Feature': features_v2,
    'Coefficient': ols_v2.coef_
})
coef_top = coef_all.reindex(coef_all['Coefficient'].abs().sort_values(ascending=False).index).head(15)

plt.figure(figsize=(10, 6))
colors = ['steelblue' if c > 0 else 'salmon' for c in coef_top['Coefficient']]
plt.barh(coef_top['Feature'][::-1], coef_top['Coefficient'][::-1], color=colors[::-1])
plt.xlabel('Coefficient (standardized)')
plt.title(f'OLS: Top 15 Coefficients out of {len(features_v2)}')
plt.tight_layout()
plt.show()

## Step 4: Lasso Regression (L1 Regularization)
With many features, Lasso's ability to zero out irrelevant coefficients becomes critical.

In [ ]:
# Lasso with default alpha — how many features survive?
lasso = Lasso(alpha=1.0, max_iter=10000)
lasso.fit(X2_train_s, y2_train)
pred_lasso = lasso.predict(X2_test_s)

mae_lasso = mean_absolute_error(y2_test, pred_lasso)
r2_lasso = r2_score(y2_test, pred_lasso)
n_nonzero = int(np.sum(lasso.coef_ != 0))

print(f"Lasso (alpha=1.0)")
print(f"  MAE:           {mae_lasso:.2f}")
print(f"  R²:            {r2_lasso:.4f}")
print(f"  Non-zero coefs: {n_nonzero} out of {len(features_v2)}")
print(f"\nSurviving features:")
for feat, coef in zip(features_v2, lasso.coef_):
    if coef != 0:
        print(f"  {feat:20s}: {coef:10.3f}")
print(f"\nZeroed out: {len(features_v2) - n_nonzero} features")

## Lasso: Effect of Alpha

In [ ]:
# Sweep alpha — watch features get eliminated
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
lasso_results = []

for a in alphas:
    m = Lasso(alpha=a, max_iter=10000)
    m.fit(X2_train_s, y2_train)
    p = m.predict(X2_test_s)
    lasso_results.append({
        'Alpha': a,
        'MAE': mean_absolute_error(y2_test, p),
        'Non-Zero Coefs': int(np.sum(m.coef_ != 0))
    })

lasso_df = pd.DataFrame(lasso_results)
print(lasso_df.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.semilogx(lasso_df['Alpha'], lasso_df['MAE'], 'o-', lw=2, ms=8)
ax1.set_xlabel('Alpha (log scale)')
ax1.set_ylabel('MAE')
ax1.set_title('Lasso: Regularization Strength vs Error')
ax1.grid(True, alpha=0.3)

ax2.semilogx(lasso_df['Alpha'], lasso_df['Non-Zero Coefs'], 's-', lw=2, ms=8, color='orange')
ax2.set_xlabel('Alpha (log scale)')
ax2.set_ylabel('Non-Zero Coefficients')
ax2.set_title(f'Lasso: Feature Selection (of {len(features_v2)} total)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Ridge Regression (L2 Regularization)
Ridge shrinks all coefficients toward zero but never eliminates them entirely.

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X2_train_s, y2_train)
pred_ridge = ridge.predict(X2_test_s)

mae_ridge = mean_absolute_error(y2_test, pred_ridge)
r2_ridge = r2_score(y2_test, pred_ridge)

print(f"Ridge (alpha=1.0)")
print(f"  MAE:           {mae_ridge:.2f}")
print(f"  R²:            {r2_ridge:.4f}")
print(f"  Non-zero coefs: {int(np.sum(ridge.coef_ != 0))} out of {len(features_v2)}  (Ridge never zeros)")

## Ridge: Effect of Alpha

In [ ]:
ridge_results = []
for a in alphas:
    m = Ridge(alpha=a)
    m.fit(X2_train_s, y2_train)
    p = m.predict(X2_test_s)
    ridge_results.append({'Alpha': a, 'MAE': mean_absolute_error(y2_test, p)})

ridge_df = pd.DataFrame(ridge_results)
print(ridge_df.to_string(index=False))

plt.figure(figsize=(8, 5))
plt.semilogx(ridge_df['Alpha'], ridge_df['MAE'], 'o-', lw=2, ms=8, color='green')
plt.xlabel('Alpha (log scale)')
plt.ylabel('MAE')
plt.title('Ridge: Regularization Strength vs Error')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 6: ElasticNet (L1 + L2)

In [ ]:
elastic = ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
elastic.fit(X2_train_s, y2_train)
pred_elastic = elastic.predict(X2_test_s)

mae_elastic = mean_absolute_error(y2_test, pred_elastic)
r2_elastic = r2_score(y2_test, pred_elastic)
n_nonzero_en = int(np.sum(elastic.coef_ != 0))

print(f"ElasticNet (alpha=1.0, l1_ratio=0.5)")
print(f"  MAE:           {mae_elastic:.2f}")
print(f"  R²:            {r2_elastic:.4f}")
print(f"  Non-zero coefs: {n_nonzero_en} out of {len(features_v2)}")

## Final Model Comparison

In [ ]:
all_results = pd.DataFrame([
    {'Model': 'OLS (price only)',       'Features': 1,                 'MAE': mae_v1,      'R²': r2_v1},
    {'Model': f'OLS ({len(features_v2)} features)', 'Features': len(features_v2), 'MAE': mae_v2,   'R²': r2_v2},
    {'Model': 'Lasso',                  'Features': n_nonzero,         'MAE': mae_lasso,   'R²': r2_lasso},
    {'Model': 'Ridge',                  'Features': len(features_v2),  'MAE': mae_ridge,   'R²': r2_ridge},
    {'Model': 'ElasticNet',             'Features': n_nonzero_en,      'MAE': mae_elastic, 'R²': r2_elastic},
])

print(all_results.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# MAE comparison
colors = ['#aaa', '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bars = ax1.bar(all_results['Model'], all_results['MAE'], color=colors, edgecolor='black', alpha=0.85)
for bar, val in zip(bars, all_results['MAE']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.0f}',
             ha='center', va='bottom', fontsize=10)
ax1.set_ylabel('MAE')
ax1.set_title('Mean Absolute Error')
ax1.tick_params(axis='x', rotation=25)
ax1.grid(axis='y', alpha=0.3)

# Features used
ax2.bar(all_results['Model'], all_results['Features'], color=colors, edgecolor='black', alpha=0.85)
ax2.set_ylabel('Features Used')
ax2.set_title('Number of Features (Lasso selects a subset)')
ax2.tick_params(axis='x', rotation=25)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Coefficient Comparison

In [ ]:
# Pick top 10 features by max absolute coefficient across all four models
all_coefs = np.column_stack([ols_v2.coef_, lasso.coef_, ridge.coef_, elastic.coef_])
max_abs = np.max(np.abs(all_coefs), axis=1)
top_idx = np.argsort(max_abs)[::-1][:10]
top_feats = [features_v2[i] for i in top_idx]

coef_compare = pd.DataFrame({
    'Feature': top_feats,
    'OLS': ols_v2.coef_[top_idx],
    'Lasso': lasso.coef_[top_idx],
    'Ridge': ridge.coef_[top_idx],
    'ElasticNet': elastic.coef_[top_idx],
})
print(coef_compare.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(top_feats))
w = 0.2
ax.bar(x - 1.5*w, coef_compare['OLS'].values,        w, label='OLS')
ax.bar(x - 0.5*w, coef_compare['Lasso'].values,      w, label='Lasso')
ax.bar(x + 0.5*w, coef_compare['Ridge'].values,      w, label='Ridge')
ax.bar(x + 1.5*w, coef_compare['ElasticNet'].values, w, label='ElasticNet')
ax.set_xticks(x)
ax.set_xticklabels(top_feats, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Coefficient (standardized)')
ax.set_title('Coefficient Comparison — Top 10 Features (by max across models)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()